# GroundedNutriRec - Notebook 01: Multi-Dataset Loading and Formatting
This notebook handles the initial loading, validation, format conversion (CSV → Parquet), and sampling for the two primary datasets in the internship roadmap:
1. **Food.com Recipes and Reviews** (Primary Recommender Benchmark)
2. **RecipeNLG Dataset** (2.2M Recipes for Language Generation & RAG Explanations)

To optimize storage and computational performance, we convert both datasets to Parquet format and export representative sub-samples (20,000 records each) for rapid prototyping.

In [1]:
import pandas as pd
import numpy as np
import os
import ast

# Define path configurations
raw_dir = os.path.join('..', 'data', 'raw')
sample_dir = os.path.join('..', 'data', 'sample')
os.makedirs(sample_dir, exist_ok=True)

print('Raw data directory path:', os.path.abspath(raw_dir))

Raw data directory path: C:\Users\Kush Shah\OneDrive\Desktop\Internship\data\raw


## Part 1: Food.com Dataset Loading & Processing
We load the recipes and reviews from the raw folder. If the parquet files are already present, we use them directly; otherwise, we read from the raw CSVs and save them as Parquet files.

In [2]:
recipes_csv = os.path.join(raw_dir, 'recipes.csv')
recipes_parquet = os.path.join(raw_dir, 'recipes.parquet')
reviews_csv = os.path.join(raw_dir, 'reviews.csv')
reviews_parquet = os.path.join(raw_dir, 'reviews.parquet')

# Process Recipes
if os.path.exists(recipes_parquet):
    print('Loading recipes from parquet...')
    df_recipes = pd.read_parquet(recipes_parquet)
elif os.path.exists(recipes_csv):
    print('Loading recipes from CSV...')
    df_recipes = pd.read_csv(recipes_csv)
    print('Saving recipes as Parquet for performance optimization...')
    df_recipes.to_parquet(recipes_parquet, index=False)
else:
    print('ERROR: Food.com recipes dataset not found!')
    df_recipes = None

# Process Reviews
if os.path.exists(reviews_parquet):
    print('Loading reviews from parquet...')
    df_reviews = pd.read_parquet(reviews_parquet)
elif os.path.exists(reviews_csv):
    print('Loading reviews from CSV...')
    df_reviews = pd.read_csv(reviews_csv)
    print('Saving reviews as Parquet for performance optimization...')
    df_reviews.to_parquet(reviews_parquet, index=False)
else:
    print('ERROR: Food.com reviews dataset not found!')
    df_reviews = None

if df_recipes is not None and df_reviews is not None:
    print(f'Food.com Recipes Shape: {df_recipes.shape}')
    print(f'Food.com Reviews Shape: {df_reviews.shape}')

Loading recipes from parquet...


Loading reviews from parquet...


Food.com Recipes Shape: (522517, 28)
Food.com Reviews Shape: (1401982, 8)


### Create Food.com Samples
We sample 20,000 recipes and their corresponding reviews to create a lightweight workspace development split.

In [3]:
if df_recipes is not None:
    sample_recipes_path = os.path.join(sample_dir, 'recipes_sample.parquet')
    sample_reviews_path = os.path.join(sample_dir, 'reviews_sample.parquet')
    
    # Randomly sample recipes
    df_recipes_sample = df_recipes.sample(n=20000, random_state=42)
    sample_recipe_ids = set(df_recipes_sample['RecipeId'])
    
    # Filter reviews matching the sampled recipe IDs
    df_reviews_sample = df_reviews[df_reviews['RecipeId'].isin(sample_recipe_ids)]
    
    # Export Parquet samples
    df_recipes_sample.to_parquet(sample_recipes_path, index=False)
    df_reviews_sample.to_parquet(sample_reviews_path, index=False)
    print(f'Saved Food.com recipe sample ({df_recipes_sample.shape}) and reviews sample ({df_reviews_sample.shape})')

Saved Food.com recipe sample ((20000, 28)) and reviews sample ((55034, 8))


## Part 2: RecipeNLG Dataset Loading & Processing
The RecipeNLG dataset contains 2.2 million recipes, which is highly useful for training RAG models. We scan the `data/raw/` directory for any file containing `recipenlg` or `dataset.csv` (excluding Food.com files) and load it. If not present, we output download and setup instructions.

In [4]:
# Search raw folder for RecipeNLG
nlg_candidates = [
    f for f in os.listdir(raw_dir)
    if ('nlg' in f.lower() or 'recipenlg' in f.lower() or f.lower() == 'dataset.csv') 
    and not f.endswith('.zip')
]

nlg_file = None
if nlg_candidates:
    nlg_file = os.path.join(raw_dir, nlg_candidates[0])
    print(f'Found RecipeNLG candidate file: {nlg_file}')
else:
    print('RecipeNLG dataset not detected in data/raw/.')
    print('→ Please download the dataset from Kaggle: https://www.kaggle.com/datasets/paultimothymooney/recipenlg')
    print('→ Place the extracted CSV file (dataset.csv) inside c:\\Users\\Kush Shah\\OneDrive\\Desktop\\Internship\\data\\raw\\ and rename it to recipenlg.csv.')

Found RecipeNLG candidate file: ..\data\raw\recipenlg.parquet


### Parsing and Converting RecipeNLG (If present)
If RecipeNLG is present, we load it, parse stringified list fields like `ingredients` and `directions`, convert it to Parquet, and save a sampled version.

In [5]:
if nlg_file is not None:
    print(f'Reading RecipeNLG file: {nlg_file}...')
    # Load a small head first to inspect schema
    if nlg_file.endswith('.parquet'):
        df_nlg_head = pd.read_parquet(nlg_file).head(5)
    else:
        df_nlg_head = pd.read_csv(nlg_file, nrows=5)
    print('Columns in RecipeNLG:', list(df_nlg_head.columns))
    
    # Convert whole file to parquet if not already done
    nlg_parquet_path = os.path.join(raw_dir, 'recipenlg.parquet')
    if not os.path.exists(nlg_parquet_path):
        print('Converting RecipeNLG CSV to Parquet format...')
        df_nlg = pd.read_csv(nlg_file)
        df_nlg.to_parquet(nlg_parquet_path, index=False)
        print('Saved recipenlg.parquet successfully!')
    else:
        df_nlg = pd.read_parquet(nlg_parquet_path)
        print('Loaded recipenlg.parquet successfully!')
        
    print('RecipeNLG Shape:', df_nlg.shape)
    
    # Create a 20,000 record sample for development
    df_nlg_sample = df_nlg.sample(n=20000, random_state=42)
    sample_nlg_path = os.path.join(sample_dir, 'recipenlg_sample.parquet')
    df_nlg_sample.to_parquet(sample_nlg_path, index=False)
    print(f'Saved RecipeNLG sample of size {df_nlg_sample.shape} to {sample_nlg_path}')
else:
    print('Skipping RecipeNLG loading (File not present). Please download the file to proceed with NLG tasks.')

Reading RecipeNLG file: ..\data\raw\recipenlg.parquet...


Columns in RecipeNLG: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']


Loaded recipenlg.parquet successfully!
RecipeNLG Shape: (2231142, 7)


Saved RecipeNLG sample of size (20000, 7) to ..\data\sample\recipenlg_sample.parquet
